# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore the [FAIR² dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

print(f"Dataset name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We will list all record sets, fields (with their `@id`s), and columns. Entities are referenced by their `@id` as per the Croissant schema.

In [ ]:
# Display all record sets and their fields, columns, and ids
from pprint import pprint

record_sets = list(dataset.metadata.record_sets)
print(f"Number of record sets: {len(record_sets)}\n")

for rec in record_sets:
    print(f"Record Set: {rec.name} (@id={rec.id})")
    print("  Fields:")
    for field in rec.fields:
        print(f"    - {field.name} (@id={field.id}, column={field.column.id if hasattr(field, 'column') and field.column is not None else None})")
    print("  Columns:")
    for col in rec.columns:
        print(f"    - {col.name} (@id={col.id})")
    print()

We now preview a sample record in the first record set using the `@id`.

In [ ]:
# Preview some records from the first record set
if record_sets:
    first_record_set_id = record_sets[0].id
    print(f"First record set id: {first_record_set_id}")
    print("Sample records from this record set:")
    for i, rec in enumerate(dataset.records(record_set=first_record_set_id)):
        pprint(rec)
        if i>=1:
            break

## 3. Data Extraction
Load data from the available record sets into DataFrames for analysis. We use the record set and field `@id`s from the overview above.

In [ ]:
# Extract data from each record set
record_set_ids = [r.id for r in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded record set: {record_set_id}, with {df.shape[0]} records and {df.shape[1]} columns.")
    else:
        print(f"No records found for record set: {record_set_id}")

if dataframes:
    # Use the first record set
    primary_record_set_id = list(dataframes.keys())[0]
    print(f"\nSelected dataframe columns ({primary_record_set_id}):")
    print(dataframes[primary_record_set_id].columns.tolist())
    display(dataframes[primary_record_set_id].head())
else:
    primary_record_set_id = None

## 4. Exploratory Data Analysis (EDA)

Let us select a numeric field for further EDA. We will refer to columns by their `@id` where possible and handle missing values sensibly (e.g., by dropping or filling them).

This example filters, normalizes, and groups by one or more fields.

In [ ]:
# Choose a numeric field and group field using field/column @id's (replace if your record set differs)
if primary_record_set_id:
    df = dataframes[primary_record_set_id]
    # Try to auto-detect numeric fields (float or int)
    numeric_fields = df.select_dtypes(include=['float', 'int']).columns.tolist()
    print(f"Numeric fields: {numeric_fields}")
    if numeric_fields:
        numeric_field = numeric_fields[0]
    else:
        # Fall back: try to find a field named like 'abundance', 'MW', etc
        candidates = [c for c in df.columns if ('abundance' in c.lower()) or ('mw' in c.lower()) or ('count' in c.lower())]
        numeric_field = candidates[0] if candidates else df.columns[0]
    print(f"Selected numeric field: {numeric_field}")

    threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
    # Filter and drop NaNs
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"\nFiltered records with {numeric_field} > {threshold:.2f}: {filtered_df.shape[0]}")
    display(filtered_df.head())

    # Normalize the field
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a string field, e.g. sample type or similar
    group_candidates = [c for c in df.columns if (df[c].dtype == 'object' and c != numeric_field)]
    if group_candidates:
        group_field = group_candidates[0]
        print(f"Grouping by: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        print(f"\nGrouped data by {group_field} (showing mean of {numeric_field}):")
        display(grouped_df.head())
    else:
        print("No suitable group-by field found.")
else:
    print("No primary record set with data available for EDA.")

## 5. Visualization

Visualize data distributions or relationships. Here, we visualize the distribution of the selected numeric field and, if available, grouped means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if primary_record_set_id and numeric_field in df:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.tight_layout()
    plt.show()

    if 'group_field' in locals():
        plt.figure(figsize=(10,4))
        grouped_df.plot(kind='bar', legend=False)
        plt.ylabel(f"Mean {numeric_field}")
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we have:

- Loaded the Croissant dataset schema and records using `mlcroissant`.
- Explored available record sets, fields, and their identifiers.
- Loaded the data into pandas DataFrames and previewed its structure.
- Applied standard EDA steps such as filtering, normalization, and grouping based on numeric and categorical fields.
- Visualized important numerical distributions and group-level summaries.

This workflow can be extended for in-depth biological or clinical analyses, feature engineering, or machine learning applications.